# Final results: MovieLens 1M

Run these cells from `Code/`. The notebook reads the recorded results; it does not download data or train models.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

results = Path("../Results")
main = pd.read_csv(results / "final/main_kpis.csv").set_index("method")
columns = ["rmse_mean", "rmse_std", "nll_mean", "brier_mean", "ece_mean",
           "fit_seconds_mean", "predict_ms_per_1k_mean"]
main[columns].round(5)

## Graph-only effect

The comparison keeps the MAP learner fixed. Positive RMSE reduction favors graph smoothing. The 95% intervals resample test users within each split, with the fitted models held fixed. The five splits overlap.

In [ ]:
ci = pd.read_csv(results / "analysis/paired_user_bootstrap.csv")
graph = ci[(ci.baseline == "MF_MAP") & (ci.comparison == "GMRF_MAP")].copy()
assert len(graph) == 5
fig, ax = plt.subplots(figsize=(7, 3))
ax.errorbar(graph.rmse_reduction, graph.seed, fmt="o",
            xerr=[graph.rmse_reduction - graph.rmse_ci_low,
                  graph.rmse_ci_high - graph.rmse_reduction], capsize=3)
ax.axvline(0, color="gray", ls="--")
ax.set(xlabel="RMSE reduction (stars); positive favors the graph",
       ylabel="Split seed", yticks=graph.seed)
plt.show()
graph[["seed", "rmse_reduction", "rmse_ci_low", "rmse_ci_high"]]

## Rating sparsity and corruption

This table compares mean RMSE across seeds 42–44. Compare paired methods within each condition.

In [ ]:
summary = pd.read_csv(results / "analysis/summary.csv")
subset = summary[summary.scenario.str.startswith(("retain_", "noise_"))]
stress = subset.pivot(index="scenario", columns="method", values="rmse_mean")
stress = stress[["MF_MAP", "GMRF_MAP"]]
stress.assign(graph_gain=stress.MF_MAP - stress.GMRF_MAP).round(5)

## Replay audit

The replay confirms reproducibility of one split. It is excluded from the five-split aggregate.

In [ ]:
import json
audit = json.loads((results / "final/seed42_replication.json").read_text())
assert audit["passed"] and audit["split_hashes_matched"]
audit